In [1]:
# first tests: background/signal
# 1. train on 80/20, test on 100/0 # test on no signal
# 2. train on 80/20, test on 0/100 # test on all signal
# 3. train on 100/0, test on 80/20 # learn on no signal
# 4. train on 0/100, test on 80/20 # learn on all signal

In [ ]:
# define data (what changes? fraction of signal/background)
# define signal and sideband regions, label data
# train model (standardize data, create tensors/datasets, optimize)
# evaluate model (test data, figures of merit--AUC, accuracy, purity, completeness)

In [ ]:
import torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from livelossplot import PlotLosses
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def toy_trainval_data(background_percent, signal_percent):
    total_events = 10000
    trainval_events = total_events * 0.9
    background_events = background_percent * trainval_events
    signal_events = signal_percent * trainval_events

    # background distribution
    Xb = np.random.uniform(low=-10, high=10, size=(background_events, 6))
    Xb = np.array(Xb, dtype=np.float32)
    yb = np.zeros((len(Xb), 1), dtype=np.float32)

    Xb_train, Xb_val, yb_train, yb_val = train_test_split(Xb, yb, test_size = 1/9) # 80% train, 10% val, 10% test

    # signal distribution
    mu_s = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
    cov_s = np.eye(6) # identity matrix for covariance
    Xs = np.random.multivariate_normal(mu_s, cov_s, size=signal_events)
    Xs = np.array(Xs, dtype=np.float32)
    ys = np.ones((len(Xs), 1), dtype=np.float32)

    Xs_train, Xs_val, ys_train, ys_val = train_test_split(Xs, ys, test_size = 1/9)

    # guarantees that training, validation, and test have the same percentage of background vs signal points
    print(f"Background training set has {len(Xb_train)} events.")
    print(f"Background validation set has {len(Xb_val)} events.")
    print(f"Signal training set has {len(Xs_train)} events.")
    print(f"Signal validation set has {len(Xs_val)} events.")

    X_train = np.vstack([Xb_train, Xs_train])
    X_val = np.vstack([Xb_val, Xs_val])
    y_train = np.concatenate([yb_train, ys_train])
    y_val = np.concatenate([yb_val, ys_val])

    idx = np.random.permutation(len(X_train))
    X_train = X_train[idx]
    y_train = y_train[idx]

    idx = np.random.permutation(len(X_val))
    X_val = X_val[idx]
    y_val = y_val[idx]

    df_signal = pd.DataFrame(Xs, columns=['p1', 'p2', 'p3', 'p4', 'p5', 'p6'])
    signal_parameter = df_signal['p6']

    p6_median = np.median(signal_parameter)
    p6_std = np.std(signal_parameter)
    print('Signal Parameter p6 median = ', p6_median)
    print('Signal Parameter p6 standard deviation = ', p6_std)

    # signal region will be defined as 1.5 standard deviations away from the median
    sig_reg_low = p6_median - (2 * p6_std)
    sig_reg_high = p6_median + (2 * p6_std)
    print('Signal Region Range: ', sig_reg_low, ' to ', sig_reg_high)

    X_train_background_region = []
    X_train_signal_region = []
    X_val_background_region = []
    X_val_signal_region = []

    for event in X_train:
        if event[5] < sig_reg_low or event[5] > sig_reg_high:
            X_train_background_region.append(event)
        else:
            X_train_signal_region.append(event)
    for event in X_val:
        if event[5] < sig_reg_low or event[5] > sig_reg_high:
            X_val_background_region.append(event)
        else:
            X_val_signal_region.append(event)

    X_train_background_region = np.array(X_train_background_region)
    X_train_signal_region = np.array(X_train_signal_region)
    X_val_background_region = np.array(X_val_background_region)
    X_val_signal_region = np.array(X_val_signal_region)

    y_train_background_region = np.zeros((len(X_train_background_region), 1), dtype=np.float32)
    y_train_signal_region = np.ones((len(X_train_signal_region), 1), dtype=np.float32)
    y_val_background_region = np.zeros((len(X_val_background_region), 1), dtype=np.float32)
    y_val_signal_region = np.ones((len(X_val_signal_region), 1), dtype=np.float32)

    X_train = np.vstack([X_train_background_region, X_train_signal_region]).astype(np.float32)
    X_val = np.vstack([X_val_background_region, X_val_signal_region]).astype(np.float32)
    y_train = np.concatenate((y_train_background_region, y_train_signal_region)).astype(np.float32)
    y_val = np.concatenate((y_val_background_region, y_val_signal_region)).astype(np.float32)

    print(f"Training set has {len(X_train)} events.")
    print(f"Validation set has {len(X_val)} events.")


def toy_test_data(background_percent, signal_percent):
    total_events = 10000
    total_test_events = 0.1 * 10000

    background_events = total_test_events * background_percent
    signal_events = total_test_events * signal_percent

    # background distribution
    Xb_test = np.random.uniform(low=-10, high=10, size=(background_events, 6))
    Xb_test = np.array(Xb_test, dtype=np.float32)
    yb_test = np.zeros((len(Xb_test), 1), dtype=np.float32)

    # signal distribution
    mu_s = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
    cov_s = np.eye(6) # identity matrix for covariance
    Xs_test = np.random.multivariate_normal(mu_s, cov_s, size=signal_events)
    Xs_test = np.array(Xs_test, dtype=np.float32)
    ys_test = np.ones((len(Xs_test), 1), dtype=np.float32)

    print(f"Background test set has {len(Xb_test)} events.")
    print(f"Signal test set has {len(Xs_test)} events.")

    X_test = np.vstack([Xb_test, Xs_test])
    y_test = np.concatenate([yb_test, ys_test])

    idx = np.random.permutation(len(X_test))
    X_test = X_test[idx]
    y_test = y_test[idx]

class NeuralNetwork(nn.Module): # class groups variables and functions together
    def __init__(self, input_dim = 5, hidden_dim = 256, output_dim = 1): # create tool, parameters for recalling, call function is used after
        super().__init__() # access all methods, attributes, etc of class
        self.NeuralNetwork = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),       
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),       
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),       
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.NeuralNetwork(x)

def train(X_train, y_train, X_val, y_val, X_test, y_test):
    ### initialize the scaler
    scaler = StandardScaler()

    ### IMPORTANT -- you only want to define your scaling ("fit_transform") based on your training dataset.
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled_5 = [] # remove 6th parameter from input
    for item in X_train_scaled:
        item = item[0:5]
        X_train_scaled_5.append(item)

    ### Now re-use the same transformation on validation & test sets ("transform")
    X_val_scaled   = scaler.transform(X_val)
    X_val_scaled_5 = []
    for item in X_val_scaled:
        item = item[0:5]
        X_val_scaled_5.append(item)

    X_test_scaled  = scaler.transform(X_test)
    X_test_scaled_5 = []
    for item in X_test_scaled:
        item = item[0:5]
        X_test_scaled_5.append(item)

    print("Train mean (before scaling):", np.array(X_train.mean(axis=0)))
    print("Train mean (after scaling) -- each dimension should be close to 0:", X_train_scaled.mean(axis=0))
    print("Train std (before scaling)", np.array(X_train.std(axis=0)))
    print("Train std (after scaling) -- each dimension should be close to 1:", X_train_scaled.std(axis=0))

    X_train_scaled_tensor = torch.tensor(np.array(X_train_scaled_5))
    y_train_tensor = torch.tensor(y_train)

    X_val_scaled_tensor = torch.tensor(np.array(X_val_scaled_5))
    y_val_tensor = torch.tensor(y_val)

    X_test_scaled_tensor = torch.tensor(np.array(X_test_scaled_5))
    y_test_tensor = torch.tensor(y_test)

    train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_scaled_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_scaled_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size =16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for inputs, labels in train_loader:
        print(inputs.shape, labels.shape)
        break

    # train and validate, plot loss and accuracy

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # requires previous check for gpu
    print(f'Using device: {device}')

    # initialize model, loss function, optimizer
    model = NeuralNetwork(input_dim = 5, hidden_dim = 256, output_dim = 1).to(device)
    loss_fn = nn.BCELoss() # common loss function for multi-class classification problems
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # default learning rate for Adam optimizer is 0.001

    liveloss = PlotLosses(figsize=(9,4))

    logs = {}

    for epoch in range(20):  # loop over the dataset multiple times (overfitting at 50 epochs)

        # training
        model.train()
        total_train_loss = 0
        train_acc, count = 0.0, 0
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # zero the parameter gradients, updates weights only on the mini batch
            inputs, labels = inputs.to(device), labels.to(device)
            out = model(inputs)  # forward pass
            loss = loss_fn(out, labels)  # compute loss

            loss.backward()  # backward pass
            optimizer.step()  # update parameters
            total_train_loss += loss.item()

            predictions = (out > 0.5).float()
            train_acc += (predictions == labels).sum().item()
            count += labels.size(0)

        train_loss_per_batch = total_train_loss / len(train_loader)
        logs['loss'] = train_loss_per_batch

        train_count_accuracy = train_acc / count
        logs['acc per count'] = train_count_accuracy


        # validation
        model.eval()
        total_val_loss = 0
        val_acc, count = 0.0, 0
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            out = model(inputs)  # forward pass
            loss = loss_fn(out, labels)  # compute loss
            total_val_loss += loss.item()

            predictions = (out > 0.5).float()
            val_acc += (predictions == labels).sum().item()
            count += labels.size(0)

        val_loss_per_batch = total_val_loss / len(val_loader)
        logs['val_loss'] = val_loss_per_batch

        val_count_accuracy = val_acc / count
        logs['val_acc per count'] = val_count_accuracy

        liveloss.update(logs)
        liveloss.send()
        print(f'Epoch {epoch}, Loss: {loss.item()}')
        print(f'Epoch {epoch}, Accuracy: {val_count_accuracy}')

    # run on test set

    model.eval()
    test_loss, test_acc, count = 0.0, 0.0, 0
    all_scores = []
    all_labels = []
    all_out = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            
            inputs = inputs.to(device).float()
            labels = labels.to(device).float()
        
            out = model(inputs)
            all_out.extend(out.squeeze().cpu().numpy())
            loss = loss_fn(out, labels)
            test_loss += loss.item() * inputs.size(0)

            scores = (out > 0.5).float()
            all_scores.extend(scores.squeeze().cpu().numpy())
            all_labels.extend(labels.squeeze().cpu().numpy())
                
            # Track counts
            test_acc += (scores == labels).sum().item()
            count += labels.size(0)

    print('Length of Assigned Scores: ', len(out))
    print('Assigned Scores: ', out)
    print('Length of Signal/Region Labels: ', len(scores))
    print('Signal/Region Labels: ', scores)

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)

    print(f"Average test loss: {test_loss/count:.4f} | Test accuracy: {test_acc/count:.3f}")

    plt.hist(all_out)

    auc = roc_auc_score(all_labels, all_scores)
    print(f"Test ROC-AUC (vs true labels): {auc:.3f}  (0.5 = no separation, 1.0 = perfect)")

    data = {"Labels": all_labels, "Predicted Values": all_scores}
    df = pd.DataFrame(data)

    # confusion matrix, compares predicted labels with actual labels
    # columns = predicted classes, rows = actual classes

    cm = confusion_matrix(all_labels, all_scores)
    cm_display = ConfusionMatrixDisplay(confusion_matrix = cm)
    cm_display.plot()
    plt.show()


In [ ]:
# 1. train on 80/20, test on 100/0 # test on no signal

toy_trainval_data
toy_test_data
test = train
